## Import Library

In [1]:
import pandas as pd
print("Everything is working!")

Everything is working!


## Data Loading & Initial Inspection

In [3]:
# Note the 'r' before the quotes—it helps Python read the backslashes \
df = pd.read_csv(r'D:\spam\spam.csv', encoding='latin-1')

# Then do the cleanup as before
df = df[['v1', 'v2']]
df.columns = ['label', 'message']
print(df.head())

  label                                            message
0   ham  Go until jurong point, crazy.. Available only ...
1   ham                      Ok lar... Joking wif u oni...
2  spam  Free entry in 2 a wkly comp to win FA Cup fina...
3   ham  U dun say so early hor... U c already then say...
4   ham  Nah I don't think he goes to usf, he lives aro...


## Text Preprocessing

In [4]:
import nltk
nltk.download('stopwords')
nltk.download('punkt')
print("Language tools downloaded!")

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\ANURAGHI\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\ANURAGHI\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt.zip.


Language tools downloaded!


In [5]:
import string
from nltk.corpus import stopwords

# Get the list of English 'stop words' (common words like 'the', 'is', 'in')
stop_words = set(stopwords.words('english'))

def transform_text(text):
    # 1. Convert to lowercase
    text = text.lower()
    
    # 2. Remove punctuation (like ! ? . ,)
    text = "".join([char for char in text if char not in string.punctuation])
    
    # 3. Remove stopwords and split into words
    text = [word for word in text.split() if word not in stop_words]
    
    return " ".join(text)

# Apply this to your 'message' column
df['message'] = df['message'].apply(transform_text)

print("Original message vs Cleaned message:")
print(df.head())

Original message vs Cleaned message:
  label                                            message
0   ham  go jurong point crazy available bugis n great ...
1   ham                            ok lar joking wif u oni
2  spam  free entry 2 wkly comp win fa cup final tkts 2...
3   ham                u dun say early hor u c already say
4   ham        nah dont think goes usf lives around though


## Text Vectorization

In [6]:
from sklearn.feature_extraction.text import CountVectorizer

# Initialize the tool
cv = CountVectorizer()

# Transform the messages into numbers (X)
X = cv.fit_transform(df['message'])

# Our target is the 'label' (y)
y = df['label']

print(f"Number of messages: {X.shape[0]}")
print(f"Number of unique words (features): {X.shape[1]}")

Number of messages: 5572
Number of unique words (features): 9376


In [7]:
df.head()

,label,message
0,ham,go jurong point crazy available bugis n great ...
1,ham,ok lar joking wif u oni
2,spam,free entry 2 wkly comp win fa cup final tkts 2...
3,ham,u dun say early hor u c already say
4,ham,nah dont think goes usf lives around though


## Train-Test Split

In [8]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# 1. Split the data (80% for training, 20% for testing)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 2. Initialize and Train the Random Forest
# We use your favorite model!
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# 3. Make predictions on the test set
y_pred = model.predict(X_test)

# 4. See how we did
print(f"Accuracy: {accuracy_score(y_test, y_pred) * 100:.2f}%")
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Accuracy: 97.04%

Confusion Matrix:
[[965   0]
 [ 33 117]]


## Real-World Testing

In [9]:
def predict_spam(sample_message):
    # 1. Clean the input just like we cleaned the training data
    cleaned_msg = transform_text(sample_message)
    # 2. Convert to numbers using our CV
    vectorized_msg = cv.transform([cleaned_msg])
    # 3. Predict!
    prediction = model.predict(vectorized_msg)
    return prediction[0]

# --- TEST IT OUT ---
test1 = "Hey, are you coming for the football match tonight?"
test2 = "CONGRATULATIONS! You have won a 1000 dollar gift card. Click here to claim now!"

print(f"Message 1 is: {predict_spam(test1)}")
print(f"Message 2 is: {predict_spam(test2)}")

Message 1 is: ham
Message 2 is: spam


In [10]:
def predict_spam(sample_message):
    cleaned_msg = transform_text(sample_message)
    vectorized_msg = cv.transform([cleaned_msg])
    prediction = model.predict(vectorized_msg)
    return prediction[0]
test1 = "Hi, u oka?"
test2 = "CONGRATULATIONS! I love you!"

print(f"Message 1 is: {predict_spam(test1)}")
print(f"Message 2 is: {predict_spam(test2)}")

Message 1 is: ham
Message 2 is: ham


## Saving the Model

In [11]:
import pickle
pickle.dump(model, open('spam_model.pkl', 'wb'))
pickle.dump(cv, open('vectorizer.pkl', 'wb'))

In [12]:
import pickle

# 1. Save the trained model to a file
with open('spam_model.pkl', 'wb') as model_file:
    pickle.dump(model, model_file)

# 2. Save the vectorizer (this is crucial because it holds your vocabulary)
with open('vectorizer.pkl', 'wb') as vec_file:
    pickle.dump(cv, vec_file)

print("Files saved successfully! Check your project folder.")

Files saved successfully! Check your project folder.


#### The model achieved 97% accuracy. The most important features (words) for spam were 'free', 'txt', and 'claim'